## Executive summary (read this first)
Review one frozen 21-method daily suite. Diagnostics reuse explored dates. The January 2024 outer period is evaluated after freezing a shortlist and has limited coverage. No official improvement is established. Monthly forecasts stay unchanged.

In [ ]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
plan = json.loads((root / 'plan.json').read_text(encoding='utf-8'))
print(plan['warning'])
print('Frozen shortlist:', json.loads((root / 'shortlist.json').read_text(encoding='utf-8'))['configs'])
display(pd.read_csv(root / 'diagnostic_summary.csv'))
display(pd.read_csv(root / 'diagnostic_by_year.csv').pivot(index='year', columns='config', values='local_relative_loss'))

In [ ]:
decision = json.loads((root / 'decision.json').read_text(encoding='utf-8'))
print(decision)
if (root / 'validation_summary.csv').exists():
    display(pd.read_csv(root / 'validation_summary.csv'))
    print(json.loads((root / 'validation_paired.json').read_text(encoding='utf-8')))
else:
    print('No later validation table; inspect decision and coverage.')

In [ ]:
for phase in ['diagnostic', 'validation']:
    case_path = root / (phase + '_cases.csv')
    if not case_path.exists():
        continue
    cases = pd.read_csv(case_path)
    assert not cases.duplicated(['config','group','origin','seed']).any()
    expected = cases.config.nunique()
    assert (cases.groupby(['group','origin','seed']).config.nunique() == expected).all()
    audit = pd.read_csv(root / (phase + '_weight_audit.csv'))
    active = audit.inner_latest_target_index.notna()
    assert (audit.loc[active,'inner_latest_target_index'] <= audit.loc[active,'inner_last_history_index']).all()
    display(audit.groupby(['config','selected_mix']).size().unstack(fill_value=0))
    components = cases.groupby(['config','cluster'])[['marginal_contribution','joint_contribution','tail_contribution','coverage90']].mean().groupby('config').mean()
    print(phase, 'equal basket component means')
    display(components)
print('Paired coverage and history audit passed. No competition submission created.')